# M6.4 — Persistent clean and particle source caches

Plan: [`plans/milestone_06/06_sequence_generation_plan.md`](../../plans/milestone_06/06_sequence_generation_plan.md).  
Next: `06_5_matrix_sweeps.ipynb`.

Demonstrates the Phase 4 cache contract independently of the historical M6.2 smoke notebook (`06_2_generate_sequences_no_cache.ipynb`).

Runs the smoke sequence twice: forced clean/particle cache `MISS`, then identical-input cache `HIT`. Diffusion solves and camera capture still run on both passes. Live Netgen/NGSolve objects, FEM spaces, matrices, operators, and solver handles are never persisted.

Outputs live under `data/generated/m6_4/` (sequences as children; `_cache/` beside them) so the cache-disabled M6.2 artifact under `m6_2/` stays untouched. Before each pass, only the planned demo sequence directory is removed; `m6_4/_cache/` is left intact.


In [1]:
from pathlib import Path

ROOT = Path.cwd()
while not (ROOT / "pyproject.toml").exists():
    if ROOT == ROOT.parent:
        raise RuntimeError("Could not locate repository root containing pyproject.toml")
    ROOT = ROOT.parent

!pip install --quiet --no-cache-dir "{ROOT}[fem,dev]" -c "{ROOT}/requirements.txt"

from gummybear.paths import display_path

print(f"ROOT={display_path(ROOT)}")



[notice] A new release of pip is available: 24.0 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
ROOT=.


In [2]:
import pandas as pd
from IPython.display import display
from pathlib import Path

import gummybear_validation.milestone_06.validation as m6_validation
from gummybear.datasets.generation_plan import (
    build_execution_plan,
    run_generation_plan,
    summarize_execution_plan,
    validate_generation_plan,
)
from gummybear.datasets.generation_workbook import load_generation_workbook
from gummybear_validation.milestone_06 import (
    clear_demo_sequence_outputs,
    load_sequence_manifest,
    miss_hit_cache_tables,
)
from gummybear_validation.notebook_tools import run_installed_pytest_test

WORKBOOK_PATH = ROOT / "configs" / "m6" / "m6_generation_plan.xlsx"
DEMO_OUTPUT_ROOT = ROOT / "data" / "generated" / "m6_4"
CACHE_ROOT = DEMO_OUTPUT_ROOT / "_cache"


In [3]:
workbook = load_generation_workbook(WORKBOOK_PATH)
plan = validate_generation_plan(workbook, repo_root=ROOT)
execution_plan = build_execution_plan(plan, limit=1, cache_root=CACHE_ROOT)
summary = summarize_execution_plan(
    execution_plan,
    disabled_sequence_count=len(plan.disabled_sequence_ids),
)
print(f"Workbook: {display_path(workbook.path)}")
print(f"Demo output root: {display_path(DEMO_OUTPUT_ROOT)}")
print(f"Cache root: {display_path(Path(execution_plan.cache_root))}")
summary.to_dict()


Workbook: configs/m6/m6_generation_plan.xlsx
Demo output root: data/generated/m6_4
Cache root: data/generated/m6_4/_cache


{'workbook_path': 'configs/m6/m6_generation_plan.xlsx',
 'workbook_sha256': '4e6ac865c398b1c7c60e91a80f90270c993e3356568f5e71f79532ba7a378fde',
 'enabled_sequence_count': 1,
 'disabled_sequence_count': 0,
 'clean_group_count': 1,
 'particle_group_count': 1,
 'diffusion_group_count': 1,
 'sequence_count': 1,
 'frame_count': 6,
 'expected_clean_cache_hits': 0,
 'expected_clean_cache_misses': 1,
 'expected_particle_cache_hits': 0,
 'expected_particle_cache_misses': 1,
 'output_roots': ('data/generated/m6_2',),
 'resolutions': ((128, 128),),
 'plans_diffusion_operator_cache': False,
 'output_status_counts': {},
 'warnings': ()}

## Forced MISS followed by cache HIT

`force_recompute=True` makes the first pass publish fresh payloads even when an earlier demonstration populated the shared `_cache`. The second pass uses identical scientific inputs and must load both source caches.

Each pass clears `m6_4/<sequence_id>/` first so delta planning cannot skip the HIT pass as a complete no-op. The scenario `_cache/` is preserved. The historical no-cache sequence under `data/generated/m6_2/bear_m6_smoke_001/` is not touched.


In [4]:
jobs = list(execution_plan.jobs)

clear_demo_sequence_outputs(jobs, DEMO_OUTPUT_ROOT, repo_root=ROOT)
first_result = run_generation_plan(
    execution_plan,
    output_root=DEMO_OUTPUT_ROOT,
    limit=1,
    max_workers=1,
    force_recompute=True,
    use_persistent_cache=True,
    verbose=True,
)

clear_demo_sequence_outputs(jobs, DEMO_OUTPUT_ROOT, repo_root=ROOT)
second_result = run_generation_plan(
    execution_plan,
    output_root=DEMO_OUTPUT_ROOT,
    limit=1,
    max_workers=1,
    use_persistent_cache=True,
    verbose=True,
)

first = first_result.generated[0]
second = second_result.generated[0]
assert first.clean_cache.status == first.particle_cache.status == "miss"
assert first.clean_cache.reason == first.particle_cache.reason == "forced_recompute"
assert second.clean_cache.status == second.particle_cache.status == "hit"


Removed prior demo sequence output: data/generated/m6_4/bear_m6_smoke_001


In [5]:
cache_status_table, timing_table = miss_hit_cache_tables(first, second)
print("Cache outcomes")
display(cache_status_table)
print("Operation timing comparison")
display(timing_table.style.format("{:.3f}"))

sequence_dir = Path(second.output_path)
manifest = load_sequence_manifest(sequence_dir)
{
    "sequence_id": manifest["sequence_id"],
    "output_path": display_path(sequence_dir),
    "manifest_cache_events": manifest["caches"]["events"],
}


Cache outcomes


,Forced MISS status,Forced MISS reason,Cache HIT status,Cache HIT reason
Clean source cache,MISS,forced_recompute,HIT,hit
Particle source cache,MISS,forced_recompute,HIT,hit


Operation timing comparison


,Forced MISS (s),Cache HIT (s),Time saved (s),Speedup (×)
Clean source,31.901,0.706,31.195,45.192
Particle source,10.002,0.001,10.001,10277.806
Diffusion solves,0.294,0.015,0.279,20.005
Camera capture,15.656,0.019,15.637,822.428
Four-stage total,57.853,0.741,57.112,78.116


{'sequence_id': 'bear_m6_smoke_001',
 'output_path': 'data/generated/m6_4/bear_m6_smoke_001',
 'manifest_cache_events': {'camera_visibility': {'cache_id': 'aggregate',
   'load_seconds': 0.0,
   'reason': 'all_hit;n_poses=6',
   'status': 'hit',
   'write_seconds': 0.0},
  'clean_optical': {'cache_id': '83b9d7a583728c38f41e3a12e1a6e68e8da19ce0e33a7c5b5d2a5fad27e6a521',
   'load_seconds': 0.003217834047973156,
   'reason': 'hit',
   'status': 'hit',
   'write_seconds': 0.0},
  'particle_source': {'cache_id': '2b71544f544171d6d73ac35abce5b8cac4de6482fa96e9afdafc4aaf762bcfd1',
   'load_seconds': 0.0008949171751737595,
   'reason': 'hit',
   'status': 'hit',
   'write_seconds': 0.0},
  'phi_sampling_localization': {'cache_id': 'aggregate',
   'load_seconds': 0.0,
   'reason': 'all_hit;n_poses=6',
   'status': 'hit',
   'write_seconds': 0.0}}}

## Installed-package cache contract check

Fast pytest check: numeric cache publication, reload, and diffusion-mesh alignment rejection without the full physics pipeline.


In [6]:
run_installed_pytest_test(
    m6_validation,
    "test_source_cache_roundtrip_and_mesh_alignment",
)


Milestone 6.4
Test executed: test_source_cache_roundtrip_and_mesh_alignment()

pytest:
../../venv/lib/python3.12/site-packages/gummybear_validation/milestone_06/validation.py . [100%]
============================== 1 passed in 1.00s ===============================

Test proves: A completed numeric source-cache pair is reusable only with matching key, schema,
             required arrays, and diffusion-mesh alignment.
